In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.model_selection import (train_test_split, cross_val_score, 
                                     cross_validate, KFold, StratifiedKFold)
from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score

In [2]:
X, y = load_breast_cancer(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, 
                                                    stratify=y, random_state=42)


In [3]:
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

single_scores = []

for seed in range(20):
    Xa, Xb, ya, yb = train_test_split(X_train, y_train, test_size=0.25, random_state=42)
    single_scores.append(model.fit(Xa, ya).score(Xb, yb))

print("Min:", min(single_scores), "Max:", max(single_scores), "Std:", np.std(single_scores))

Min: 0.9912280701754386 Max: 0.9912280701754386 Std: 0.0


In [4]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="accuracy")
print("Fold scores: ", cv_scores.round(3))
print(f"Mean = {cv_scores.mean():.3f}, Std = {cv_scores.std():.3f}")

Fold scores:  [0.967 0.989 0.978 0.989 0.967]
Mean = 0.978, Std = 0.010


In [5]:
Xi, yi = make_classification(n_samples=1000, weights=[0.95, 0.05], random_state=42)
for name, splitter in [("KFold", KFold(n_splits=5)), 
                       ("StratifiedKFold", StratifiedKFold(n_splits=5))]:
    positives = [int(yi[val].sum()) for _, val in splitter.split(Xi, yi)]
    print(f"{name:<16} positives in each validation fold:", positives)

KFold            positives in each validation fold: [9, 10, 11, 10, 13]
StratifiedKFold  positives in each validation fold: [10, 10, 11, 11, 11]


In [6]:
fold_scores = []
for fold, (tr_idx, va_idx) in enumerate(cv.split(X_train, y_train), start=1):
    m= clone(model).fit(X_train[tr_idx], y_train[tr_idx])
    acc = accuracy_score(y_train[va_idx], m.predict(X_train[va_idx]))
    fold_scores.append(acc)
    print(f"Fold {fold}: train rows = {len(tr_idx)}, validation rows = {len(va_idx)}, accuracy = {acc:.3f}")
print("Mean:", np.mean(fold_scores))

Fold 1: train rows = 364, validation rows = 91, accuracy = 0.967
Fold 2: train rows = 364, validation rows = 91, accuracy = 0.989
Fold 3: train rows = 364, validation rows = 91, accuracy = 0.978
Fold 4: train rows = 364, validation rows = 91, accuracy = 0.989
Fold 5: train rows = 364, validation rows = 91, accuracy = 0.967
Mean: 0.9780219780219781


##### Wrong selection of features

In [8]:
rng = np.random.default_rng(0)
X_noise = rng.normal(size =(100, 2000))
y_noise = rng.integers(0, 2, size=100)

X_leaky = SelectKBest(f_classif, k=20).fit_transform(X_noise, y_noise)
leaky = cross_val_score(LogisticRegression(max_iter=1000), X_leaky, y_noise, cv=5)

In [9]:
pipe = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=1000))
correct = cross_val_score(pipe, X_noise, y_noise, cv=5)

print("Leak mean score :", leaky.mean())
print("Correct mean score :", correct.mean())

Leak mean score : 0.78
Correct mean score : 0.5900000000000001


##### Generalization

In [12]:
for name, mdl in[("Logistic Regression", model),
                  ("Unrestricted decision tree", DecisionTreeClassifier(random_state=42))]:
    res = cross_validate(mdl, X_train, y_train, cv=cv,
                         scoring=["accuracy", "f1"], return_train_score=True)
    print(f"{name:<28} train acc = {res["train_accuracy"].mean():.3f} |"
          f"val acc = {res["test_accuracy"].mean():.3f} | "
          f"val f1 = {res["test_f1"].mean():.3f}")

    

Logistic Regression          train acc = 0.989 |val acc = 0.978 | val f1 = 0.983
Unrestricted decision tree   train acc = 1.000 |val acc = 0.916 | val f1 = 0.932


In [18]:
candidates = {"Logistic Regression": model,
              "Decision Tree": DecisionTreeClassifier(random_state=42),
              "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42)}

In [19]:
for name, mdl in candidates.items():
    s=cross_val_score(mdl, X_train, y_train, cv=cv, scoring="f1")
    print(f"{name:<20} F1 = {s.mean():.3f} ± {s.std():.3f}")

Logistic Regression  F1 = 0.983 ± 0.008
Decision Tree        F1 = 0.932 ± 0.014
Random Forest        F1 = 0.970 ± 0.015


In [21]:
best = candidates["Logistic Regression"]
best.fit(X_train, y_train)
print("Final test accuracy:", best.score(X_test, y_test))


Final test accuracy: 0.9824561403508771
